# AT2017gfo en las bandas de Roman

Sí, la recopilación es de **Villar et al. 2017** (ApJL 851, L21): su tabla 3 homogeniza la
fotometría de 18 papers de AT2017gfo y conserva la procedencia de cada punto.

Por cada noche de los primeros 5 días se junta **toda** la fotometría publicada de esa noche, se
convierte a las bandas de Roman y se le pregunta al clasificador. Las noches con más cobertura
llegan a 16 filtros de 18 instrumentos.

Esto no es un resultado del clasificador — las métricas están medidas sobre el test split en
`training/evaluation.ipynb`. Es la comparación con lo que ya se observó.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

REPOSITORY_ROOT = Path.cwd().parents[1] if Path.cwd().name == "validation" else Path.cwd()
sys.path.insert(0, str(REPOSITORY_ROOT / "training"))
sys.path.insert(0, str(REPOSITORY_ROOT / "notebooks/validation"))

from observed_photometry import (  # noqa: E402
    CONVERSION_METHODS,
    classify_windows,
    epoch_photometry,
    known_filters,
    load_classifier,
    load_villar_photometry,
    perturb_magnitudes,
    pivot_wavelength_aa,
    single_epoch_window,
)

from kilonova.photometry.roman_noise import bands_observed_at_visit, build_tier_constants  # noqa: E402
from kilonova.photometry.spectra import ALL_ROMAN_BANDS  # noqa: E402

AT2017GFO_DIR = REPOSITORY_ROOT / "data/at2017gfo"
VILLAR_CACHE = AT2017GFO_DIR / "villar2017_table3.csv"

HOST_REDSHIFT = 0.0098  # NGC 4993
MAXIMUM_PHASE_DAYS = 5.0
NIGHT_HALF_WIDTH_DAYS = 0.25  # media noche a cada lado: junta el seguimiento de una misma noche
NIGHT_PHASE_GRID = np.round(np.arange(0.5, MAXIMUM_PHASE_DAYS + 0.01, 0.5), 2)
MEASUREMENT_REALIZATIONS = 64

CHECKPOINT = REPOSITORY_ROOT / "training/checkpoints/kilonova_transformer-soup.ckpt"
NORMALIZATION_FILE = REPOSITORY_ROOT / "data/openuniverse/normalization.json"

# Las triadas que puede ver una primera visita de Roman. La cadencia del HLTDS observa la banda
# ancla del juego mas dos de las otras cuatro, asi que hay cuatro combinaciones posibles; cual toca
# depende de donde caiga el merger en el ciclo de visitas, que es una moneda al aire. Se derivan de
# `bands_observed_at_visit`, que es la regla que armo el set de entrenamiento.
FIRST_VISIT_TRIADS = {}
for tier_name in ("wide", "deep"):
    constants = build_tier_constants(tier_name)
    for parity in (0, 1):
        triad = bands_observed_at_visit(parity, constants["bands"], constants["anchor_band"])
        FIRST_VISIT_TRIADS[" ".join(triad)] = (constants, parity)

villar_photometry = load_villar_photometry(VILLAR_CACHE)
villar_photometry = villar_photometry[villar_photometry["phase"] <= MAXIMUM_PHASE_DAYS]
print(f"{len(villar_photometry)} medidas, {villar_photometry['reference'].nunique()} papers")
print("triadas: " + " | ".join(FIRST_VISIT_TRIADS))


378 medidas, 18 papers
triadas: R062 Z087 Y106 | R062 J129 H158 | Z087 Y106 J129 | Z087 H158 F184


## De los filtros publicados a las bandas de Roman

Cada magnitud AB se convierte en un $F_\nu$ en la longitud de onda pivote de su filtro (de la
librería de `pyphot`), la SED entre puntos es una ley de potencias por tramos, y esa SED se integra
contra los mismos bandpasses de `galsim.roman` con los que se armó el set de entrenamiento.

**Sin extrapolar.** El espectro interpolado abarca solo entre el pivote más azul y el más rojo
medidos esa noche, así que una banda de Roman que se salga de ahí vuelve como no observada en vez de
como un número inventado. Por eso hace falta mezclar instrumentos: ningún equipo óptico solo cubre
R062 *y* Z087 — R062 pide un filtro más azul que 4550 Å y Z087 uno más rojo que 10250 Å, y el más
rojo del óptico ($y$) se queda en 9630 Å.

Cada realización sortea las magnitudes publicadas de sus errores **antes** de convertir, y después
sortea el ruido de Roman.


In [2]:
def nightly_photometry(long_photometry, phase_grid, half_width):
    """Una noche = toda la fotometria publicada de esa noche, sin importar quien la tomo."""
    usable = long_photometry[long_photometry["filter"].isin(known_filters(long_photometry["filter"]))]
    nights = []
    for phase_days in phase_grid:
        near = usable[np.abs(usable["phase"] - phase_days) <= half_width]
        magnitudes = epoch_photometry(near, phase_days, half_width)
        if len(magnitudes) < 2:
            continue
        roman_magnitudes = CONVERSION_METHODS["sed"](magnitudes)
        nights.append(
            {
                "phase_days": float(phase_days),
                "magnitudes": magnitudes,
                "instruments": sorted(set(near["instrument"])),
                "references": sorted(set(near["reference"])),
                "covered": [band for band in ALL_ROMAN_BANDS if np.isfinite(roman_magnitudes[band])],
            }
        )
    return nights


def covers(night, triad):
    """La noche mide las TRES bandas de la triada.

    No es una exigencia estetica. En el millon de ventanas del entrenamiento la primera epoca tiene
    exactamente tres bandas observadas, siempre: la regla de cadencia pone la banda ancla en todas
    las visitas y otras dos rotan. Una noche que no cubre alguna de las tres deja ese slot como token
    `n`, que es una configuracion que el modelo no vio nunca, y la respuesta no significa nada -- de
    hecho sube a 1.00000, porque el hueco se parece a una kilonova sin flujo azul. Se reporta "-"."""
    return set(triad.split()).issubset(night["covered"])


def classify_nights(nights, triads, classifier, device, parquet_path, n_realizations):
    """P(KN) mediana sin redshift de cada noche en cada triada que la noche cubre entera."""
    random_generator = np.random.default_rng(0)
    windows = []
    for night in nights:
        for triad, (constants, parity) in triads.items():
            if not covers(night, triad):
                continue
            for realization in range(n_realizations):
                drawn = CONVERSION_METHODS["sed"](
                    perturb_magnitudes(night["magnitudes"], random_generator)
                )
                window = single_epoch_window(
                    f"{triad}|{night['phase_days']:04.1f}|{realization:04d}", drawn, HOST_REDSHIFT,
                    constants, noise_seed=realization, parity=parity,
                )
                if window is not None:
                    windows.append(window)
    predictions = classify_windows(
        pd.concat(windows, ignore_index=True), parquet_path, classifier, device,
        epochs=1, has_redshift=False,
    )
    predictions["key"] = predictions["object_id"].str.rsplit("|", n=1).str[0]
    median = predictions.groupby("key")["kn_probability"].median()

    rows = []
    for night in nights:
        row = {
            "día": night["phase_days"],
            "filtros": " ".join(sorted(night["magnitudes"], key=pivot_wavelength_aa)),
            "inst.": len(night["instruments"]),
            "papers": len(night["references"]),
            "bandas Roman": " ".join(night["covered"]),
        }
        for triad in triads:
            key = f"{triad}|{night['phase_days']:04.1f}"
            row[triad] = (
                round(float(median[key]), 5) if covers(night, triad) else "—"
            )
        rows.append(row)
    return pd.DataFrame(rows)


device = "cuda" if torch.cuda.is_available() else "cpu"
classifier, _ = load_classifier(CHECKPOINT, NORMALIZATION_FILE, device)

nights = nightly_photometry(villar_photometry, NIGHT_PHASE_GRID, NIGHT_HALF_WIDTH_DAYS)
results = classify_nights(
    nights, FIRST_VISIT_TRIADS, classifier, device,
    AT2017GFO_DIR / "at2017gfo_nightly_windows.parquet", MEASUREMENT_REALIZATIONS,
)
results.to_csv(AT2017GFO_DIR / "at2017gfo_nightly_datasets.csv", index=False)

probabilities = pd.to_numeric(
    results[list(FIRST_VISIT_TRIADS)].stack(), errors="coerce"
).dropna()
print(
    f"{len(nights)} noches, {len(probabilities)} de las "
    f"{len(nights) * len(FIRST_VISIT_TRIADS)} combinaciones con la triada completa; "
    f"P(KN) minimo {probabilities.min():.5f}"
)
display(results)


10 noches, 33 de las 40 combinaciones con la triada completa; P(KN) minimo 0.99966


,día,filtros,inst.,papers,bandas Roman,R062 Z087 Y106,R062 J129 H158,Z087 Y106 J129,Z087 H158 F184
0,0.5,U g V r i z Y y J H Ks,11,10,R062 Z087 Y106 J129 H158 F184,0.99972,0.9998,0.99978,0.99981
1,1.0,U B g V r R i I J H Ks,7,8,R062 Z087 Y106 J129 H158 F184,0.99972,0.99984,0.99981,0.99982
2,1.5,u U B g V r R i I z Y y J H K Ks,18,13,R062 Z087 Y106 J129 H158 F184,0.99974,0.99986,0.99981,0.99983
3,2.0,B g V r R i I z J H Ks,6,5,R062 Z087 Y106 J129 H158 F184,0.99966,0.99986,0.99978,0.99981
4,2.5,u B g V r R i I z Y y J H K Ks,15,11,R062 Z087 Y106 J129 H158 F184,0.99972,0.99988,0.99979,0.99981
5,3.0,g V r R i I z J H Ks,5,5,Z087 Y106 J129 H158 F184,—,—,0.99978,0.9998
6,3.5,u B g V r R i I z Y J H K Ks,13,11,R062 Z087 Y106 J129 H158 F184,0.99978,0.99989,0.99978,0.9998
7,4.0,r R I J H Ks,4,4,Z087 Y106 J129 H158 F184,—,—,0.99976,0.99979
8,4.5,B g V r R i I z Y J H K Ks,10,10,R062 Z087 Y106 J129 H158 F184,0.9998,0.9999,0.99976,0.99978
9,5.0,r R I F110W F160W,3,4,Z087 Y106 J129,—,—,0.99977,—


## El párrafo

Las 10 noches de los primeros 5 días, en cada tríada que la noche cubre entera: **P(KN) ≥ 0.9996 en
las 33 combinaciones**. Siete de las diez noches constriñen las seis bandas de Roman.

> We tested the classifier on AT2017gfo, the only kilonova with a well-sampled early light curve.
> For each of the ten nights in the first five days after the merger we combined all published
> broad-band photometry of that night — up to 16 filters from 18 instruments, drawn from the
> 18-paper compilation of Villar et al. (2017) — and converted it to the Roman bands by
> interpolating the observed SED through the measured pivot wavelengths and integrating it against
> the same `galsim.roman` bandpasses used to build the training set, with no extrapolation beyond
> the bluest and reddest filters measured. Seven of the ten nights constrain all six Roman bands.
> A first Roman visit observes three filters, so each night was classified in every three-filter
> combination it constrains completely (33 of the 40 night-triad pairs). All are classified as
> kilonovae, with P(KN) >= 0.9996 from a single visit and without a host redshift. AT2017gfo carries
> E(B-V) = 0.105 of Galactic extinction and lies at z = 0.0098, below the redshift floor of the
> training grid, so this is an out-of-distribution check rather than a measure of performance;
> classification metrics are reported on the test split in Section X.

**Por qué las siete casillas vacías.** En el millón de ventanas del entrenamiento la primera época
tiene **exactamente tres bandas observadas, siempre** — la cadencia del HLTDS pone la banda ancla en
todas las visitas y otras dos rotan. Una noche que no cubre alguna banda de la tríada dejaría ese
slot como token `n`, una configuración que el modelo nunca vio.

El token `n` no está codificado como un límite superior: lleva su propio embedding de tipo,
`magnitude = 0` y `magnitude_mask = 0`, mientras que `u` lleva la magnitud límite de 5σ con
`magnitude_mask = 1`. Pero medido sobre la noche +1.5 d en la tríada R062 Z087 Y106, el resultado es
el mismo de todas formas:

| variante | 1 − P(KN) |
|---|---|
| tríada completa | 2.59 × 10⁻⁴ |
| sin Z087 → `n` | 2.71 × 10⁻⁴ |
| sin Y106 → `n` | 2.77 × 10⁻⁴ |
| sin R062 (ancla) → `n` | **2.62 × 10⁻⁶** |
| R062 a 30 mag (`u` genuino) | **2.50 × 10⁻⁶** |

Un `n` en una banda no-ancla mueve un 5 %; en el ancla salta un factor 100 y **cae a un 5 % del caso
de no detección real**. La red no tiene nada aprendido para "ancla ausente" y su respuesta coincide
con la del caso que sí conoce — sin flujo azul, o sea muy roja, o sea kilonova. Reportar eso sería
confundir un agujero observacional con física, así que esas casillas van como "—".
